In [5]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
import matplotlib.pyplot as plt

df = pd.read_csv('Task 3 and 4_Loan_Data.csv')

# Dropping customer_id as it is not a feature
df = df.drop(columns=['customer_id'])

# Features and target
X = df.drop(columns=['default'])
y = df['default']

print(X.shape)
print(y.value_counts())

(10000, 6)
default
0    8149
1    1851
Name: count, dtype: int64


In [6]:
# 80% train, 20% test — stratified to preserve 18.5% default ratio in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) 
X_test_scaled  = scaler.transform(X_test)        

In [7]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

# Evaluate
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]  # PD for each loan

print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1630
           1       1.00      0.99      1.00       370

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000

ROC-AUC Score: 1.0000


In [8]:
def expected_loss(credit_lines_outstanding, loan_amt_outstanding,
                  total_debt_outstanding, income,
                  years_employed, fico_score,
                  recovery_rate=0.10):

    # Build feature array in same column order as training data
    features = pd.DataFrame([[credit_lines_outstanding,
                               loan_amt_outstanding,
                               total_debt_outstanding,
                               income,
                               years_employed,
                               fico_score]],
                            columns=X.columns)

    # Scale using the SAME scaler fitted on training data
    features_scaled = scaler.transform(features)

    # Get probability of default
    pd_prob = model.predict_proba(features_scaled)[0][1]

    # Expected Loss = PD × LGD × EAD
    lgd = 1 - recovery_rate   # Loss Given Default = 90%
    ead = loan_amt_outstanding # Exposure at Default

    el = pd_prob * lgd * ead

    print(f"Probability of Default : {pd_prob:.4f}")
    print(f"Loan Amount (EAD)      : ${ead:,.2f}")
    print(f"Loss Given Default     : {lgd:.0%}")
    print(f"Expected Loss          : ${el:,.2f}")

    return round(el, 2)

In [9]:
# High risk borrower — many credit lines, high debt, low FICO
print("=== HIGH RISK BORROWER ===")
expected_loss(credit_lines_outstanding=5,
              loan_amt_outstanding=5000,
              total_debt_outstanding=20000,
              income=25000,
              years_employed=1,
              fico_score=480)

print()

# Low risk borrower — no extra credit lines, low debt, high FICO
print("=== LOW RISK BORROWER ===")
expected_loss(credit_lines_outstanding=0,
              loan_amt_outstanding=5000,
              total_debt_outstanding=3000,
              income=90000,
              years_employed=8,
              fico_score=780)

=== HIGH RISK BORROWER ===
Probability of Default : 1.0000
Loan Amount (EAD)      : $5,000.00
Loss Given Default     : 90%
Expected Loss          : $4,500.00

=== LOW RISK BORROWER ===
Probability of Default : 0.0000
Loan Amount (EAD)      : $5,000.00
Loss Given Default     : 90%
Expected Loss          : $0.00


np.float64(0.0)